# Transfer to NAS (round-robin drives)

Run this notebook **during acquisition**, alongside HAL/Dave, when using
`round_robin_drives` mode (`prepare_imaging/03`'s `DATA_DRIVES`).

Unlike `02_round_scheduler.ipynb`'s transfer step -- which only moves a round
once its mosaics have already been built **locally** (`tracker.is_round_done`)
-- this notebook transfers a round's raw data (plus the small `round_info.csv`
/ `round_bit_color_map.csv` / `positions_*.txt` / `settings/*.xml` files a
cluster-side `ExperimentMetadata.load()` needs alongside it) as soon as HAL
has **finished writing it**, with no dependency on any local QC analysis.
This is the intended companion to moving QC analysis off the microscope
computer entirely and onto a SLURM cluster instead (see
`07_cluster_submit_analysis.ipynb`) -- `01_fov_scheduler.ipynb`/
`02_round_scheduler.ipynb` don't need to run at all in that workflow.

`TRANSFER_DEST` ends up a full mirror of `SAMPLE_DIR`: each round's raw data
lands nested exactly as it is locally (`data/cells`, `data/hybs/H01`, ...,
`data/transit` if present -- see `transfer.relative_to_data_root`, which
strips a round-robin round's drive-specific prefix so it lands at the same
logical `data/...` path regardless of which physical drive it came from),
`metadata/`/`positions/`/`settings/` sync continuously every tick, and a
one-time step (section 4) mirrors the static `data/mosaic10x`/`MERci`/
`merlin`/`fishtank` folders alongside it.

Each tick:
- Syncs the small metadata/positions/settings files to `TRANSFER_DEST`, and
  rewrites the `round_info.csv` copy's `dir` column from each round's
  original absolute, drive-specific path to the same relative `data/...`
  sub-path its data was copied to -- so a cluster-side
  `ExperimentMetadata.load()` resolves paths under its own `data_dir`
  instead of a microscope-local drive that doesn't exist there (see
  `transfer.rewrite_round_info_dirs`).
- For every round that is **fully written** (every expected raw file exists
  on disk) and **not on the drive HAL is actively writing right now**, starts
  a background transfer of that round's data directory to `TRANSFER_DEST`.
- Marks each round transferred with a zero-byte sentinel once its copy
  succeeds (retried automatically on the next tick if it fails).

Requires `ANALYSIS_MODE = "round_robin_drives"` (the "which drive is hot right
now" signal this relies on) and a real `TRANSFER_DEST`.

## 1 — Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config   import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.progress        import ProgressTracker
from MERci.scheduler       import TransferScheduler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

Edit the cells below to match your experiment. `TRANSFER_DEST` is required
(unlike `01`/`02`, where it's optional) -- this notebook exists only to
transfer.

In [ ]:
SAMPLE_NAME = SAMPLE_DIR.name

# ── Image file format (must match what HAL writes) ───────────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── Transfer destination (required) ─────────────────────────────
TRANSFER_DEST = None   # e.g. r"\\NAS\experiments\LT027"

if TRANSFER_DEST is None:
    raise ValueError("Set TRANSFER_DEST to the NAS destination root before running this notebook.")

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Image suffix : {IMAGE_SUFFIX}")
print(f"Transfer dest: {TRANSFER_DEST}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    analysis_mode  = "round_robin_drives",
    transfer_dest  = TRANSFER_DEST,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

print(f"Rounds       : {meta.n_rounds}")
print(f"FOVs         : {meta.n_fovs}")
print(f"Data roots   : {config.all_data_roots}")
print(f"Transfer dest: {config.transfer_dest}")

## 3 — Check current progress

A per-round FOV tally — not just a "fully written yes/no" — so partial
progress (or a round that looks done in Explorer but isn't recognized as
such) is visible directly. Run this cell at any time.

In [ ]:
import pandas as pd

def round_progress_table(meta):
    """One row per round: how many of its expected raw files already exist on disk."""
    rows = []
    for rid in meta.valid_round_ids():
        files     = meta.files_for_round(rid)
        n_total   = len(files)
        n_present = sum(1 for f in files if f.exists())
        rows.append({
            "round_id":      rid,
            "fovs_present":  n_present,
            "fovs_expected": n_total,
            "fully_written": n_present == n_total and n_total > 0,
        })
    return pd.DataFrame(rows)

progress = round_progress_table(meta)
print(progress.to_string(index=False))

n_written     = int(progress["fully_written"].sum())
n_transferred = sum(1 for r in meta.valid_round_ids() if tracker.is_round_transferred(r))
print(f"\nRounds fully written : {n_written} / {len(progress)}")
print(f"Rounds transferred   : {n_transferred} / {len(progress)}")

### 3b — Troubleshooting: inspect one round in detail

If a round shows fewer FOVs present than you can actually see on disk in
Explorer, run this with that round's id — it prints exactly which
filename/directory MERci is looking for (from `round_info.csv`'s series
pattern) side-by-side with what's *actually* in that directory, so a
naming/path mismatch (rather than a genuinely-still-imaging round) is
immediately visible.

In [ ]:
def diagnose_round(meta, round_id, sample_fov=0, list_limit=15):
    """Compare what MERci expects for round_id against what's actually on disk."""
    files     = meta.files_for_round(round_id)
    n_present = sum(1 for f in files if f.exists())
    print(f"Round {round_id}: {n_present} / {len(files)} expected files present.\n")

    for s in meta.series_for_round(round_id):
        print(f"  series pattern : {s.name}")
        print(f"  candidate dirs : {[str(d) for d in s.candidate_dirs]}")
        expected_name = s.build_filename(sample_fov, meta.image_suffix)
        print(f"  expected filename for fov {sample_fov}: {expected_name}")
        resolved = s.resolve_path(sample_fov, meta.image_suffix)
        print(f"  resolves to    : {resolved}  (exists={resolved.exists()})")

        for cand_dir in s.candidate_dirs:
            if not Path(cand_dir).exists():
                print(f"  {cand_dir} does not exist (yet) on this machine.")
                continue
            actual = sorted(p.name for p in Path(cand_dir).iterdir())
            print(f"  actual contents of {cand_dir} ({len(actual)} entries, first {list_limit}):")
            for name in actual[:list_limit]:
                print("    ", name)
        print()

    missing = [f for f in files if not f.exists()]
    print(f"  first missing expected paths (of {len(missing)} total missing):")
    for f in missing[:10]:
        print("   ", f)


# Change round_id to inspect any round that looks stuck, e.g. 1 (cells) or 2 (H01)
diagnose_round(meta, round_id=1)

## 4 — One-time sync: mosaic10x, MERci, merlin, fishtank

These don't change during acquisition (`data/mosaic10x` is captured once
before imaging starts; `MERci`/`merlin`/`fishtank` are static config written
by the `prepare_imaging` notebooks), so they're not part of the continuous
per-round tick loop below -- run this cell once (or re-run any time; it's
additive/incremental, safe to repeat) to mirror them alongside the raw
round data, so `TRANSFER_DEST` ends up a full copy of `SAMPLE_DIR`.

`metadata/`, `positions/`, `settings/` are **not** listed here -- those are
already synced continuously, every tick, by the scheduler below.

In [ ]:
from MERci.transfer import mirror_dir_sync

EXTRA_SYNC_DIRS = [
    SAMPLE_DIR / "data" / "mosaic10x",
    SAMPLE_DIR / "MERci",
    SAMPLE_DIR / "merlin",
    SAMPLE_DIR / "fishtank",
]

for src in EXTRA_SYNC_DIRS:
    if not src.exists():
        print(f"skip (not present): {src}")
        continue
    dst = Path(TRANSFER_DEST) / src.relative_to(SAMPLE_DIR)
    print(f"syncing {src}  ->  {dst} ...")
    ok = mirror_dir_sync(src, dst)
    print("  done." if ok else "  FAILED -- see log output above.")

## 5 — Transfer scheduler

**Run this cell and leave it running** during acquisition.

Interrupt the kernel (`■` button) to stop the loop; any in-flight transfer
thread is a daemon thread and will simply be abandoned (the round stays
"not transferred" and is retried the next time this notebook runs).

In [ ]:
from datetime import timedelta

def _format_eta(seconds):
    if seconds is None:
        return "n/a (no round transfer has completed yet)"
    return str(timedelta(seconds=round(seconds)))

def show_tick(tick):
    from IPython.display import clear_output
    clear_output(wait=True)
    active_rid = meta.actively_writing_round()
    active_drv = meta.drive_of_round(active_rid) if active_rid is not None else None
    print(f"[tick {tick['iteration']}] started {tick['transfers_started']} transfer(s) this tick.")
    print(f"Active round / drive: {active_rid} / {active_drv or 'none'}")

    n_written     = sum(1 for r in meta.valid_round_ids() if meta.round_fully_written(r))
    n_transferred = sum(1 for r in meta.valid_round_ids() if tracker.is_round_transferred(r))
    print(f"Rounds written / transferred: {n_written} / {n_transferred}  (of {meta.n_rounds})")

    # `scheduler` (created below, after this function is defined) is looked up
    # at call time, once run_loop has actually started ticking.
    progress = scheduler.transfer_progress()
    print(f"FOVs transferred: {progress['fovs_arrived']} / {progress['fovs_total']}")

    avg_rate = progress["avg_seconds_per_fov"]
    if avg_rate is not None:
        print(f"Avg transfer time / FOV: {avg_rate:.1f} s  (mean over up to the last 20 completed round transfers)")
    else:
        print("Avg transfer time / FOV: not yet measured (no round transfer has completed)")

    if progress["in_progress_rounds"]:
        print(
            f"Current round(s) {progress['in_progress_rounds']}: "
            f"{progress['in_progress_fovs_remaining']} FOV(s) remaining, "
            f"ETA to finish: {_format_eta(progress['eta_in_progress_s'])}"
        )

    print(
        f"All remaining FOVs: {progress['all_remaining_fovs']}, "
        f"ETA to finish everything (assuming all FOVs already written): "
        f"{_format_eta(progress['eta_all_remaining_s'])}"
    )

scheduler = TransferScheduler(config, meta, tracker)
scheduler.run_loop(on_tick=show_tick)